# Scraping Masters Shot Videos using our Previous Participants

Importing appropriate Libraries

In [6]:
import os
import requests
import time
import re
import asyncio
import pandas as pd
import numpy as numpy
from playwright.async_api import async_playwright

Load in the masters past participants data

In [2]:
masters_players_df = pd.read_csv("C:/Personal Projects/finalized_masters_participants.csv")

masters_players_df.head(10)

,Year,Pos,Name,R1,R2,R3,R4,Total Score,Total Par,Made_Cut
0,2025,1,R McIlroy,72.0,66.0,66.0,73.0,277.0,-11.0,1
1,2025,2,J Rose,65.0,71.0,75.0,66.0,277.0,-11.0,1
2,2025,3,P Reed,71.0,70.0,69.0,69.0,279.0,-9.0,1
3,2025,4,S Scheffler,68.0,71.0,72.0,69.0,280.0,-8.0,1
4,2025,5,S Im,71.0,70.0,71.0,69.0,281.0,-7.0,1
5,2025,5,B DeChambeau,69.0,68.0,69.0,75.0,281.0,-7.0,1
6,2025,7,L Åberg,68.0,73.0,69.0,72.0,282.0,-6.0,1
7,2025,8,X Schauffele,73.0,69.0,70.0,71.0,283.0,-5.0,1
8,2025,8,Z Johnson,72.0,74.0,66.0,71.0,283.0,-5.0,1
9,2025,8,J Day,70.0,70.0,71.0,72.0,283.0,-5.0,1


Now going to the "Master's Vault" we can see that there is only footage for final round shots from 1968 onwards
This means we need to filter out all data prior to 1968 from our table

In [3]:
vault_players = masters_players_df[masters_players_df["Year"]>=1968]

vault_players["Year"].min()

np.int64(1968)

Perfect! It looks like we've dropped all players before 1968. Now what we want to do is create a new loop:
where for i in each row (players), we go to the vault webpage: https://www.masters.com/vault
Once there we will iterate a vault video search through each player, each year if applicable for that player, and each hole that year for said player.
I.e (Tiger Woods 1968 Hole 1-Tiger Woods 2025 Hole 18), and scrape the associated video which comes up and append it into a dataframe of videos
consisting of four columns: player, year, hole #, and the video itself. 

In [4]:
#My output path
output_path = "C:/Personal Projects/Masters_Vault_Data.csv"


In [7]:
#Create a file in my path with these headers for my data
if not os.path.exists(output_path):
    pd.DataFrame(columns=["player", "year", "hole", "shot", "video_id"]).to_csv(output_path, index=False)

url = "https://www.masters.com/apps/vault/api/search"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.masters.com",
    "Referer": "https://www.masters.com/vault/results.html",
    "User-Agent": "Mozilla/5.0",
    "Authorization": "Bearer YOUR_TOKEN_HERE"
}

#Track player years
completed = set()

if os.path.exists(output_path):
    existing_df = pd.read_csv(output_path)

    grouped = existing_df.groupby(["player", "year"])["hole"].nunique()

    for (player, year), hole_count in grouped.items():
        if hole_count == 18:
            completed.add((player, year))

for _, row in vault_players.iterrows():
    player = row["Name"]
    year = row["Year"]

    #Skip player year combinations that have already been done
    if (player, year) in completed:
        print(f"Skipping already processed: {player} {year}")
        continue

    #If a player has played in the final round for that respective year
    if pd.notna(row["R4"]):

        player_results = []

        for hole in range(1, 19):

            query = f"{player} {year} hole {hole}"
            payload = {"query": query}

            try:
                response = requests.post(
                    url,
                    json=payload,
                    headers=headers,
                    timeout=10
                )

                if response.status_code != 200:
                    print(f"Failed request: {player} {year} hole {hole}")
                    continue

                data = response.json()

                for item in data.get("docs", []):
                    title = item.get("clip_title", "")
                    vid_id = item.get("id")
                    full_name = item.get("full_name")
                    vid_year = item.get("video_year")

                    hole_match = re.search(r'Hole No\. (\d+)', title)
                    shot_match = re.search(r'Shot (\d+)', title)

                    hole_num = int(hole_match.group(1)) if hole_match else None
                    shot_num = int(shot_match.group(1)) if shot_match else None

                    player_results.append({
                        "player": full_name,
                        "year": int(vid_year) if vid_year else year,
                        "hole": hole_num,
                        "shot": shot_num,
                        "video_id": vid_id
                    })

                time.sleep(0.3)

            except Exception as e:
                print(f"Error with {player} {year} hole {hole}: {e}")
                continue

        #Save all player data for a single year as long as all 18 holes are collected
        if player_results:
            df_chunk = pd.DataFrame(player_results).drop_duplicates()

            #If there's any partial rows of incomplete player data delete them before replacing with new data
            if os.path.exists(output_path):
                existing_df = pd.read_csv(output_path)

                existing_df = existing_df[
                    ~((existing_df["player"] == player) & (existing_df["year"] == year))
                ]

                existing_df.to_csv(output_path, index=False)

            #Append to our existing vault df
            df_chunk.to_csv(
                output_path,
                mode="a",
                header=False,
                index=False
            )
            #Tell me the player and the year 
            print(f"✅ Saved: {player} {year} | {len(df_chunk)} rows")
        #Tell me if there was an error with a specific player/year combo
        else:
            print(f"⚠️ No data found: {player} {year}")


✅ Saved: R McIlroy 2025 | 76 rows
✅ Saved: J Rose 2025 | 39 rows
✅ Saved: P Reed 2025 | 17 rows
✅ Saved: S Scheffler 2025 | 29 rows
✅ Saved: S Im 2025 | 2 rows
✅ Saved: B DeChambeau 2025 | 71 rows
✅ Saved: L Åberg 2025 | 458 rows
✅ Saved: X Schauffele 2025 | 4 rows
✅ Saved: Z Johnson 2025 | 3 rows
✅ Saved: J Day 2025 | 13 rows
✅ Saved: C Conners 2025 | 23 rows
✅ Saved: H English 2025 | 458 rows
✅ Saved: M Homa 2025 | 20 rows
✅ Saved: B Watson 2025 | 6 rows
✅ Saved: J Rahm 2025 | 18 rows
✅ Saved: J Spieth 2025 | 10 rows
⚠️ No data found: T Hatton 2025
✅ Saved: M McCarty 2025 | 3 rows
✅ Saved: T Hoge 2025 | 2 rows
✅ Saved: C Morikawa 2025 | 5 rows
✅ Saved: H Matsuyama 2025 | 25 rows
⚠️ No data found: D Riley 2025
✅ Saved: T Fleetwood 2025 | 3 rows
✅ Saved: D Berger 2025 | 11 rows
✅ Saved: B An 2025 | 458 rows
✅ Saved: V Hovland 2025 | 8 rows
✅ Saved: A Rai 2025 | 458 rows
✅ Saved: M Kim 2025 | 8 rows
✅ Saved: S Theegala 2025 | 6 rows
✅ Saved: D McCarthy 2025 | 3 rows
✅ Saved: J Niemann 2